In [1]:
import os
from PIL import Image
import albumentations as A
import numpy as np
import cv2

# Paths to folders
optical_flow_folder = "/data/maryam.sana/vimeo_unicontrol/Uni-ControlNet/data/UVG/optical_flow/Jockey"
encoded_frame_folder = "/data/maryam.sana/vimeo_unicontrol/Uni-ControlNet/data/UVG/Jockey/quality_8"
target_folder = "/data/maryam.sana/vimeo_unicontrol/Uni-ControlNet/data/UVG/Jockey"

# Load image file names
optical_flow_files = sorted([os.path.join(optical_flow_folder, f) for f in os.listdir(optical_flow_folder) if f.endswith('.png')])[4:10]
encoded_frame_files = sorted([os.path.join(encoded_frame_folder, f) for f in os.listdir(encoded_frame_folder) if f.endswith('.png')])[4:10]
target_files = sorted([os.path.join(target_folder, f) for f in os.listdir(target_folder) if f.endswith('.png')])[4:10]

# Ensure all folders have the same number of images
assert len(optical_flow_files) == len(encoded_frame_files) == len(target_files), "Mismatch in number of images across folders."

# Define augmentations
augmentations = A.Compose(
    [
        A.RandomCrop(height=512, width=512),
        A.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.4,
            p=1.0
        ),
        A.Rotate(limit=10, border_mode=cv2.BORDER_REFLECT, p=1.0),
        A.Affine(scale=(0.8, 1.2), translate_percent=(0.2, 0.2), p=1.0),
    ],
    additional_targets={"encoded_frame": "image", "target_image": "image"},
)

# Function to preprocess, augment, and save images
def preprocess_augment_and_save(optical_flow, encoded_frame, target_image, size=(512, 512)):
    """Preprocesses, applies augmentations, and saves augmented images."""
    optical_flow = np.array(Image.open(optical_flow).resize(size).convert('RGB'))
    encoded_frame = np.array(Image.open(encoded_frame).resize(size).convert('RGB'))
    target_image = np.array(Image.open(target_image).resize(size).convert('RGB'))

    augmented = augmentations(image=optical_flow, encoded_frame=encoded_frame, target_image=target_image)

    # Save augmented images
    def save_image(image_array, save_path):
        """Saves an image to the given path."""
        aug_image = Image.fromarray(image_array)
        aug_image.save(save_path)

    # Generate augmented file paths
    optical_flow_aug_path = os.path.splitext(optical_flow_files[0])[0] + "_aug.png"
    encoded_frame_aug_path = os.path.splitext(encoded_frame_files[0])[0] + "_aug.png"
    target_image_aug_path = os.path.splitext(target_files[0])[0] + "_aug.png"

    # Save augmented images
    save_image(augmented["image"], optical_flow_aug_path)
    save_image(augmented["encoded_frame"], encoded_frame_aug_path)
    save_image(augmented["target_image"], target_image_aug_path)

    print(f"Augmented images saved: {optical_flow_aug_path}, {encoded_frame_aug_path}, {target_image_aug_path}")


# Process all files
for optical_flow, encoded_frame, target_image in zip(optical_flow_files, encoded_frame_files, target_files):
    preprocess_augment_and_save(optical_flow, encoded_frame, target_image)
 

Augmented images saved: /data/maryam.sana/vimeo_unicontrol/Uni-ControlNet/data/UVG/optical_flow/Jockey/im00006_aug.png, /data/maryam.sana/vimeo_unicontrol/Uni-ControlNet/data/UVG/Jockey/quality_8/im00005_aug.png, /data/maryam.sana/vimeo_unicontrol/Uni-ControlNet/data/UVG/Jockey/im00005_aug.png
Augmented images saved: /data/maryam.sana/vimeo_unicontrol/Uni-ControlNet/data/UVG/optical_flow/Jockey/im00006_aug.png, /data/maryam.sana/vimeo_unicontrol/Uni-ControlNet/data/UVG/Jockey/quality_8/im00005_aug.png, /data/maryam.sana/vimeo_unicontrol/Uni-ControlNet/data/UVG/Jockey/im00005_aug.png
Augmented images saved: /data/maryam.sana/vimeo_unicontrol/Uni-ControlNet/data/UVG/optical_flow/Jockey/im00006_aug.png, /data/maryam.sana/vimeo_unicontrol/Uni-ControlNet/data/UVG/Jockey/quality_8/im00005_aug.png, /data/maryam.sana/vimeo_unicontrol/Uni-ControlNet/data/UVG/Jockey/im00005_aug.png
Augmented images saved: /data/maryam.sana/vimeo_unicontrol/Uni-ControlNet/data/UVG/optical_flow/Jockey/im00006_aug.

In [3]:
import os
import random
import cv2
import numpy as np

from torch.utils.data import Dataset

from src.train.util import *

class UniDataset(Dataset):
    def __init__(self,
                 anno_path,
                 image_dir,
                 condition_root,
                 local_type_list,
                 global_type_list,
                 resolution,
                 drop_txt_prob,
                 keep_all_cond_prob,
                 drop_all_cond_prob,
                 drop_each_cond_prob,
                 augmentations,
                ):
        
        file_ids, self.annos = read_anno(anno_path)
        self.image_paths = [os.path.join(image_dir, file_id + '.png') for file_id in file_ids]
        self.local_paths = {}
        for local_type in local_type_list:
            self.local_paths[local_type] = [os.path.join(condition_root, local_type, file_id + '.png') for file_id in file_ids]
        self.global_paths = {}
        for global_type in global_type_list:
            self.global_paths[global_type] = [os.path.join(condition_root, global_type, file_id + '.npy') for file_id in file_ids]
        
        self.local_type_list = local_type_list
        self.global_type_list = global_type_list
        self.resolution = resolution
        self.drop_txt_prob = drop_txt_prob
        self.keep_all_cond_prob = keep_all_cond_prob
        self.drop_all_cond_prob = drop_all_cond_prob
        self.drop_each_cond_prob = drop_each_cond_prob
        self.augmentations = augmentations
        # self.num_augmentations = num_augmentations
        self.data = []
        self.__load_data__()

    def __load_image__(self,path):
         image = cv2.imread(path)
         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
         image = cv2.resize(image, (self.resolution, self.resolution))
         return image

    def __load_data__(self):
        aug_data = [] 
        for index in range(0,len(self.annos)):
            # print('start loading')
            image = self.__load_image__(self.image_paths[index])
            augment_input = {"image": image}
            local_conditions = {}
            for local_type in self.local_type_list:
                local_conditions[local_type] = self.__load_image__(self.local_paths[local_type][index])
                augment_input[local_type] = local_conditions[local_type]
                
            global_conditions = {}  
            for global_type in self.global_type_list:
                global_conditions[global_type] = np.load(self.global_paths[global_type][index])

            if self.augmentations:
                augmented_data = self.augmentations(**augment_input)
                augmented_local = {lt: augmented_data[lt] for lt in self.local_type_list}
                aug_data.append({'image': augmented_data['image'], 
                                  'local': augmented_local, 
                                  'global': global_conditions, 
                                  'text': self.annos[index],
                                  'path': self.image_paths[index],
                                  'aug': True})


            self.data.append({'image':image,
                              'local': local_conditions,
                              'global': global_conditions,
                              'text': self.annos[index],
                              'path': self.image_paths[index],
                              'aug': False})
        print('aug done')
        self.data.extend(aug_data)
        print('augmented data loading finished', self.__len__())


    def print_info(self,index):
        print('error:' ,self.data[index]['path'],self.data[index]['aug'])

        
    def __len__(self):
        return len(self.data)
  
    
    def __getitem__(self, index):
        data = self.data[index]
        
        try:
            image = data['image']
            image = (image.astype(np.float32) / 127.5) - 1.0

        except:
            self.print_info(index)
        
        anno = data['text']        

        local_conditions = []
        for condition in data['local']:
            try:    
                condition = condition.astype(np.float32) / 255.0
                local_conditions.append(condition)
            except:
                self.print_info(index)

        global_conditions = data['global']

        if random.random() < self.drop_txt_prob:
            anno = ''
        local_conditions = keep_and_drop(local_conditions, self.keep_all_cond_prob, self.drop_all_cond_prob, self.drop_each_cond_prob)
        global_conditions = keep_and_drop(global_conditions, self.keep_all_cond_prob, self.drop_all_cond_prob, self.drop_each_cond_prob)
        if len(local_conditions) != 0:
            local_conditions = np.concatenate(local_conditions, axis=2)
        if len(global_conditions) != 0:
            global_conditions = np.concatenate(global_conditions)

        return_data = dict(jpg=image,
                    txt=anno,
                    local_conditions=local_conditions,
                    global_conditions=global_conditions,
                )
                
        # Print the shapes for debugging
        print(f"Image shape: {image.shape if image is not None else 'None'}")
        print(f"Local conditions shape: {local_conditions.shape if local_conditions is not None else 'None'}")
        print(f"Global conditions shape: {global_conditions.shape if global_conditions is not None else 'None'}")
        
        return return_data

In [ ]:
from albumentations import Compose, RandomCrop, ColorJitter, Rotate, Affine
from albumentations.pytorch import ToTensorV2

# Define consistent augmentations
augmentations = Compose(
    [
        RandomCrop(height=512, width=512),  # Random cropping
        ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.2, p=1.0),  # Adjust color properties
        Rotate(limit=10, border_mode=cv2.BORDER_REFLECT, p=1.0),  # Random rotation
        Affine(scale=(0.9, 1.1), translate_percent=(0.1, 0.1), p=1.0),  # Random affine
    ],
    additional_targets={"optical_flow": "image", "quality_8": "image"}
)

dataset = UniDataset(
    anno_path="/data2/local_datasets/vimeo_all/annotations.txt",
    image_dir="/data2/local_datasets/vimeo_all/images",
    condition_root="/data2/local_datasets/vimeo_all/condition",
    local_type_list=["optical_flow", "quality_8"],
    augmentations=augmentations,
    global_type_list=[],
    resolution=512,
    drop_txt_prob=0.5,
    keep_all_cond_prob=0.1,
    drop_all_cond_prob=0.1,
    drop_each_cond_prob=[0.5]
)

/data/maryam.sana/anaconda3/envs/unicontrol/lib/python3.8/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: libtorch_cuda_cu.so: cannot open shared object file: No such file or directory
  warn(f"Failed to load image Python extension: {e}")
